# Step 4 — GDN (GatedDeltaNet / linear_attn) INT8 Quantization


In [ ]:
# ── Paths & constants 
import os, json, shutil
import torch
from safetensors import safe_open
from safetensors.torch import save_file
from compressed_tensors.compressors import pack_to_int32, unpack_from_int32

SRC  = "/home/prashant.takale/icml/4_models/qwen3.5-4b-w8a16-clean"   # Steps 1-3 output
BASE = "/home/prashant.takale/icml/4_models/qwen-weights"             # FP16 base (GDN source)
OUT  = "/home/prashant.takale/icml/4_models/qwen3.5-4b-final-combo"   # final output

GROUP, NUM_BITS, QMAX = 128, 8, 127

# All 5 GDN projection types.
# qkv+z form one fused group; b+a form another; out_proj is standalone.

GDN_PROJ_SUFFIXES = [
    "linear_attn.in_proj_qkv",  # fused → in_proj_qkvz
    "linear_attn.in_proj_z",    # fused → in_proj_qkvz
    "linear_attn.in_proj_b",    # fused → in_proj_ba
    "linear_attn.in_proj_a",    # fused → in_proj_ba
    "linear_attn.out_proj",     # standalone
]

assert os.path.exists(SRC),  f"Steps 1-3 output not found: {SRC}"
assert os.path.exists(BASE), f"Base FP16 not found: {BASE}"
print("Paths OK.", flush=True)

In [ ]:
# ── Quantization helper 

def quant_group_sym_int8(w):
    """Symmetric per-group(128) INT8. Returns (int8_q, bf16_scale) or None if
    the input dim is not divisible by group size."""
    w = w.float()
    out, inp = w.shape
    if inp % GROUP != 0:
        return None
    ng    = inp // GROUP
    wg    = w.reshape(out, ng, GROUP)
    amax  = wg.abs().amax(dim=2, keepdim=True)
    scale = (amax / QMAX).clamp(min=1e-8)
    q     = torch.round(wg / scale).clamp(-QMAX - 1, QMAX).reshape(out, inp).to(torch.int8)
    return q, scale.reshape(out, ng).to(torch.bfloat16)

In [ ]:
# ── Load helper 

def load_all(d):
    idx = os.path.join(d, "model.safetensors.index.json")
    t = {}
    if os.path.exists(idx):
        for fn in sorted(set(json.load(open(idx))["weight_map"].values())):
            with safe_open(os.path.join(d, fn), "pt") as f:
                for k in f.keys(): t[k] = f.get_tensor(k)
    else:
        with safe_open(os.path.join(d, "model.safetensors"), "pt") as f:
            for k in f.keys(): t[k] = f.get_tensor(k)
    return t

In [ ]:
# ── Load SRC (W8A16 main + INT8 MTP) and BASE (FP16, GDN source)

print("Loading SRC (W8A16 main + INT8 MTP) ...", flush=True)
m = load_all(SRC)
print(f"  {len(m)} tensors", flush=True)

print("Loading BASE (FP16, for GDN projections) ...", flush=True)
base = load_all(BASE)
print(f"  {len(base)} tensors", flush=True)

In [ ]:
# ── Quantize all GDN projections 

targets = [
    k for k in base
    if k.endswith(".weight") and any(s in k for s in GDN_PROJ_SUFFIXES)
]
print(f"GDN projection weights to INT8: {len(targets)}", flush=True)

quantized, skipped, removed_from_ignore = 0, 0, []

for wkey in targets:
    mod = wkey[:-len(".weight")]
    res = quant_group_sym_int8(base[wkey])
    if res is None:
        skipped += 1
        print(f"  skip {mod} (input dim not divisible by {GROUP})")
        continue

    q, scale = res
    packed = pack_to_int32(q.to(torch.int8), NUM_BITS, packed_dim=1)

    # Verify roundtrip — never ship corrupt packing
    assert torch.equal(
        unpack_from_int32(packed, NUM_BITS, list(base[wkey].shape)), q
    ), f"Roundtrip verification failed for {mod}"

    if wkey in m:
        del m[wkey]   # remove any old FP16 copy
    m[f"{mod}.weight_packed"] = packed.contiguous()
    m[f"{mod}.weight_scale"]  = scale.contiguous()
    m[f"{mod}.weight_shape"]  = torch.tensor(list(base[wkey].shape), dtype=torch.int64)
    removed_from_ignore.append(mod)
    quantized += 1

print(f"\nQuantized {quantized} GDN weights, skipped {skipped}", flush=True)

In [ ]:
# ── Create output dir and copy non-tensor files from SRC 

os.makedirs(OUT, exist_ok=True)
for fn in os.listdir(SRC):
    if fn.endswith(".safetensors") or fn.endswith(".index.json"):
        continue   # will be written fresh below
    s = os.path.join(SRC, fn)
    if os.path.isfile(s):
        shutil.copy2(s, os.path.join(OUT, fn))
print("Non-tensor files copied.", flush=True)

In [ ]:
# ── Clone+contiguous all tensors before saving

# CRITICAL: safetensors zeroes tensors that share storage or are views.
# weight_shape int64 descriptors are tiny and prone to this bug — cloning fixes it.

m = {k: v.clone().contiguous() for k, v in m.items()}

print(f"Saving {len(m)} tensors to {OUT}/model.safetensors ...", flush=True)
save_file(m, os.path.join(OUT, "model.safetensors"), metadata={"format": "pt"})
print("Tensors saved.", flush=True)

In [ ]:
# ── Update config: remove quantized GDN modules from the quant ignore list ──

cfg = json.load(open(os.path.join(SRC, "config.json")))
ig  = set(cfg["quantization_config"]["ignore"])
before = len(ig)
for mod in removed_from_ignore:
    ig.discard(mod)
cfg["quantization_config"]["ignore"] = sorted(ig)
json.dump(cfg, open(os.path.join(OUT, "config.json"), "w"), indent=2)
print(f"Ignore list: {before} → {len(ig)} (removed {before - len(ig)} GDN modules)", flush=True)
print(f"\n Step 4 DONE → {OUT}", flush=True)

In [ ]:
# ── Verify final weight counts 

with safe_open(os.path.join(OUT, "model.safetensors"), "pt") as f:
    ks = list(f.keys())

main_packed = sum(1 for k in ks if "language_model" in k and k.endswith(".weight_packed"))
mtp_packed  = sum(1 for k in ks if k.startswith("mtp.") and k.endswith(".weight_packed"))
gdn_packed  = sum(1 for k in ks if "linear_attn" in k and k.endswith(".weight_packed"))

print(f"Main body INT8 packed : {main_packed}")
print(f"MTP head  INT8 packed : {mtp_packed}  (want 7)")
print(f"GDN       INT8 packed : {gdn_packed}  (want 5 types × 24 layers = 120)")